In [ ]:
import _collections_abc
from asyncio import base_events
from dotenv import load_dotenv
from pathlib import Path  
envPath = Path.cwd().parent.parent.joinpath("env").joinpath("dev.aws.env")

print(envPath)

if envPath.exists():
    load_dotenv(dotenv_path= envPath, override=True)
    print("env loaded")
else:
    print("Env file missing")

/Users/vj/Sites/python-practice-combined/notebooks
env loaded


In [6]:
from asyncio import coroutines
from botocore.exceptions import ClientError
# 1. Ensure MiniStack is running on port 4566
import boto3
from gradio_client.client import Endpoint
import os

endpoint_url = os.getenv("AWS_URL")
region = os.getenv("REGION")
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")

print(os.getenv("APP_NAME"), endpoint_url, region, aws_access_key_id, aws_secret_access_key)


None None None None None


In [30]:
# 2. Point boto3 to local endpoint
s3 = boto3.client('s3',
    endpoint_url=endpoint_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

list_of_buckets = ["my-local-bucket", "my-local-bucket-2", "test", "test-bucket"]
try:
    for bucket in list_of_buckets:
        # 3. Code logic interacts with Ministack instantly!
        s3.create_bucket(Bucket=bucket)
        print(f"✓ Successfully created bucket :: {bucket}")
except ClientError as ce:
    print(ce.response)

✓ Successfully created bucket :: my-local-bucket
✓ Successfully created bucket :: my-local-bucket-2
✓ Successfully created bucket :: test
✓ Successfully created bucket :: test-bucket


In [ ]:
response = s3.list_objects_v2(Bucket='my-local-bucket')
print(response)
if 'Contents' in response:
    for obj in response['Contents']:
        print(obj['Key'])

{'ResponseMetadata': {'RequestId': '4e2351a5-ec26-48e3-8e95-f1434b28fc24', 'HTTPStatusCode': 200, 'HTTPHeaders': {'content-type': 'application/xml', 'x-amz-request-id': '4e2351a5-ec26-48e3-8e95-f1434b28fc24', 'x-amz-id-2': 'B0LbXkPNQzaDvtGJJbUKaeufkivPC0cCIBE7AMhVyq/w8KrjD2+WNM/KemQ80CRM', 'access-control-allow-origin': '*', 'access-control-allow-methods': 'GET, POST, PUT, DELETE, HEAD, OPTIONS, PATCH', 'access-control-allow-headers': '*', 'access-control-expose-headers': '*', 'x-amzn-requestid': '4e2351a5-ec26-48e3-8e95-f1434b28fc24', 'content-length': '271', 'date': 'Sat, 04 Jul 2026 07:12:29 GMT', 'server': 'hypercorn-h11'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Name': 'my-local-bucket', 'Prefix': '', 'MaxKeys': 1000, 'EncodingType': 'url', 'KeyCount': 0}


In [29]:

# Edit these variables as needed
file_name = 'test_upload.txt'
bucket_name = 'my-local-bucket'
object_name = 'uploaded_test_file.txt'

# Create a dummy file if it doesn't exist for demonstration
if not os.path.exists(file_name):
    with open(file_name, 'w') as f:
        f.write('Hello, this is a test file for S3 upload!')
    print(f"Created temporary file: {file_name}")

try:
    print(f"Uploading {file_name} to bucket {bucket_name} as {object_name}...")
    s3.upload_file(file_name, bucket_name, object_name)
    print("✓ Upload successful!")
except Exception as e:
    print(f"Error uploading file: {e}")

Uploading test_upload.txt to bucket my-local-bucket as uploaded_test_file.txt...
✓ Upload successful!


In [31]:
bucket_name = 'my-local-bucket'

try:
    print(f"Listing files in bucket '{bucket_name}':")
    response = s3.list_objects_v2(Bucket=bucket_name)
    if 'Contents' in response:
        for obj in response['Contents']:
            print(obj)
            print(f"- {obj['Key']} ({obj['Size']} bytes)")
    else:
        print("Bucket is empty.")
except Exception as e:
    print(f"Error listing files: {e}")

Listing files in bucket 'my-local-bucket':
{'Key': 'uploaded_test_file.txt', 'LastModified': datetime.datetime(2026, 7, 4, 7, 35, 49, 252000, tzinfo=tzutc()), 'ETag': '"a93bf4ced8a5bec1f4744aa0885f006d"', 'Size': 41, 'StorageClass': 'STANDARD'}
- uploaded_test_file.txt (41 bytes)


In [22]:
bucket_name = 'my-local-bucket'
object_name = 'uploaded_test_file.txt'

try:
    print(f"Deleting '{object_name}' from bucket '{bucket_name}'...")
    s3.delete_object(Bucket=bucket_name, Key=object_name)
    print("✓ Successfully deleted object!")
except Exception as e:
    print(f"Error deleting object: {e}")

Deleting 'uploaded_test_file.txt' from bucket 'my-local-bucket'...
✓ Successfully deleted object!


In [42]:
import requests
bucket_name = 'my-local-bucket'
object_name = 'uploaded_test_file.txt'
file_url = f"{endpoint_url}/{bucket_name}/{object_name}"
print(file_url)
content = requests.get(file_url)

content.text

http://localhost:4566/my-local-bucket/uploaded_test_file.txt


'Hello, this is a test file for S3 upload!'